In [39]:
from sentence_transformers import SentenceTransformer

embedding_model_name = "intfloat/multilingual-e5-small"

encoder = SentenceTransformer(
    embedding_model_name,
    device="cpu",
)

print("Embedding dimensions:", encoder.get_embedding_dimension())
print("Maximum input tokens:", encoder.max_seq_length)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7140.07it/s]


Embedding dimensions: 384
Maximum input tokens: 512


In [40]:
example_texts = [
    "How can I solve a quadratic equation?",
    "如何求解一元二次方程？",
    "How do I bake a chocolate cake?",
]

encoder_inputs = ["query: " + text for text in example_texts]

example_embeddings = encoder.encode(
    encoder_inputs,
    normalize_embeddings=True,
)

print("Embedding matrix shape:", example_embeddings.shape)
print("First prompt's first 8 values:", example_embeddings[0, :8])

Embedding matrix shape: (3, 384)
First prompt's first 8 values: [ 0.11408317 -0.02869101 -0.05076684 -0.03300026  0.08711493 -0.03484631
  0.00548495  0.01735642]


In [41]:
import pandas as pd

similarities = example_embeddings @ example_embeddings.T

example_labels = ["English math", "Chinese math", "Cooking"]

similarity_table = pd.DataFrame(
    similarities,
    index=example_labels,
    columns=example_labels,
)

similarity_table.round(3)

,English math,Chinese math,Cooking
English math,1.000,0.845,0.829
Chinese math,0.845,1.000,0.754
Cooking,0.829,0.754,1.000


In [ ]:
from pathlib import Path
import pandas as pd
%pip install pyarrow
saved_data_dir = Path(
    "/Users/shaoguanhua/Desktop/fall-2026-right-for-less"
    "1_data_preparation/data/processed/char_tfidf_50k"
)

train_questions = pd.read_pickle(
    saved_data_dir / "train_questions.pkl"
)

print("Training rows:", len(train_questions))
print("First question:", train_questions["question_text"].iloc[0])

Note: you may need to restart the kernel to use updated packages.
Training rows: 21900
First question: 猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。

古字刚分开，豆字挤进来，你先别生气，看后笑开怀。


In [43]:
first_text = train_questions["question_text"].iloc[0]
first_input = "query: " + first_text

first_tokenized = encoder.tokenizer(
    first_input,
    add_special_tokens=True,
    truncation=False,
)

first_token_ids = first_tokenized["input_ids"]

print("Input tokens:", len(first_token_ids))
print("Encoder limit:", encoder.max_seq_length)
print(
    "First 15 tokens:",
    encoder.tokenizer.convert_ids_to_tokens(first_token_ids[:15]),
)

Input tokens: 86
Encoder limit: 512
First 15 tokens: ['<s>', '▁que', 'ry', ':', '▁', '猜', '字', '谜', ',', '根据', '我', '给', '的', '描述', '猜']


In [44]:
def count_input_tokens(text):
    tokenized = encoder.tokenizer(
        "query: " + text,
        add_special_tokens=True,
        truncation=False,
    )
    return len(tokenized["input_ids"])

train_token_counts = train_questions["question_text"].apply(
    count_input_tokens
)

over_limit = train_token_counts > encoder.max_seq_length

print(train_token_counts.describe())
print("Prompts exceeding the limit:", over_limit.sum())
print("Percentage exceeding the limit:", f"{over_limit.mean():.2%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1075 > 512). Running this sequence through the model will result in indexing errors


count    21900.000000
mean       199.328584
std        105.022070
min         14.000000
25%        103.000000
50%        180.000000
75%        288.000000
max       1075.000000
Name: question_text, dtype: float64
Prompts exceeding the limit: 120
Percentage exceeding the limit: 0.55%


In [45]:
longest_index = train_token_counts.idxmax()
longest_text = train_questions.loc[longest_index, "question_text"]

truncated_input = encoder.tokenizer(
    "query: " + longest_text,
    add_special_tokens=True,
    truncation=True,
    max_length=encoder.max_seq_length,
)

kept_text = encoder.tokenizer.decode(
    truncated_input["input_ids"],
    skip_special_tokens=True,
)

print("Sample ID:", train_questions.loc[longest_index, "sample_id"])
print("Original tokens:", train_token_counts.loc[longest_index])
print("Retained tokens:", len(truncated_input["input_ids"]))
print("\nOriginal prompt ending:\n", longest_text[-500:])
print("\nTruncated prompt ending:\n", kept_text[-500:])

Sample ID: mmlu-college-medicine.val.50
Original tokens: 1075
Retained tokens: 512

Original prompt ending:
 ed during moderate- to vigorous-intensity exercise. In fact, sauna use has been proposed as an alternative to exercise for people who are unable to engage in physical activity due to chronic disease or physical limitations.[13]

Based on the article, what would be an important thing for a person to do after sauna use?
A) Eat a meal.
B) Shower in cold water.
C) Exercise.
D) Replenish fluids with filtered water.
Print only a single choice  from "A" or "B" or "C" or "D" without explanation. Answer:

Truncated prompt ending:
 es per week had a 66 percent lower risk of developing dementia and a 65 percent lower risk of developing Alzheimer's disease, compared to men who used the sauna only one time per week... The health benefits associated with sauna use extended to other aspects of mental health, as well. Men participating in the KIHD study who used the sauna four to seven times pe

In [46]:
content_ids = encoder.tokenizer(
    longest_text,
    add_special_tokens=False,
    truncation=False,
)["input_ids"]

chunk_size = 480

long_prompt_chunks = [
    encoder.tokenizer.decode(content_ids[start:start + chunk_size])
    for start in range(0, len(content_ids), chunk_size)
]

chunk_token_counts = [
    len(encoder.tokenizer("query: " + chunk)["input_ids"])
    for chunk in long_prompt_chunks
]

print("Number of chunks:", len(long_prompt_chunks))
print("Tokens per chunk:", chunk_token_counts)
print("Final chunk ending:\n", long_prompt_chunks[-1][-500:])

assert max(chunk_token_counts) <= encoder.max_seq_length

Number of chunks: 3
Tokens per chunk: [485, 485, 115]
Final chunk ending:
 use has been proposed as an alternative to exercise for people who are unable to engage in physical activity due to chronic disease or physical limitations.[13] Based on the article, what would be an important thing for a person to do after sauna use? A) Eat a meal. B) Shower in cold water. C) Exercise. D) Replenish fluids with filtered water. Print only a single choice from "A" or "B" or "C" or "D" without explanation. Answer:


In [47]:
import numpy as np

chunk_inputs = ["query: " + chunk for chunk in long_prompt_chunks]

chunk_embeddings = encoder.encode(
    chunk_inputs,
    normalize_embeddings=True,
)

mean_chunk_embedding = chunk_embeddings.mean(axis=0)

long_prompt_embedding = (
    mean_chunk_embedding / np.linalg.norm(mean_chunk_embedding)
)

print("Chunk embedding shape:", chunk_embeddings.shape)
print("Combined prompt shape:", long_prompt_embedding.shape)
print("Combined vector length:", np.linalg.norm(long_prompt_embedding))

Chunk embedding shape: (3, 384)
Combined prompt shape: (384,)
Combined vector length: 1.0


In [48]:
def prepare_encoder_inputs(text):
    prefixed_text = "query: " + text

    full_ids = encoder.tokenizer(
        prefixed_text,
        truncation=False,
    )["input_ids"]

    if len(full_ids) <= encoder.max_seq_length:
        return [prefixed_text]

    content_ids = encoder.tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    chunk_size = 480

    chunks = [
        encoder.tokenizer.decode(content_ids[start:start + chunk_size])
        for start in range(0, len(content_ids), chunk_size)
    ]

    inputs = ["query: " + chunk for chunk in chunks]

    for chunk_input in inputs:
        token_count = len(encoder.tokenizer(chunk_input)["input_ids"])
        assert token_count <= encoder.max_seq_length

    return inputs

In [49]:
print("Short prompt inputs:", len(prepare_encoder_inputs(first_text)))
print("Long prompt inputs:", len(prepare_encoder_inputs(longest_text)))

Short prompt inputs: 1
Long prompt inputs: 3


In [50]:
def embed_prompt(text):
    inputs = prepare_encoder_inputs(text)

    chunk_vectors = encoder.encode(
        inputs,
        normalize_embeddings=True,
    )

    prompt_vector = chunk_vectors.mean(axis=0)
    vector_length = np.linalg.norm(prompt_vector)

    assert np.isfinite(vector_length) and vector_length > 0

    return prompt_vector / vector_length

In [51]:
short_prompt_embedding = embed_prompt(first_text)
recreated_long_embedding = embed_prompt(longest_text)

print("Short prompt shape:", short_prompt_embedding.shape)
print("Long prompt shape:", recreated_long_embedding.shape)
print(
    "Matches earlier long-prompt vector:",
    np.allclose(recreated_long_embedding, long_prompt_embedding),
)

Short prompt shape: (384,)
Long prompt shape: (384,)
Matches earlier long-prompt vector: True


In [52]:
train_encoder_inputs = []
train_input_owners = []

for question_position, text in enumerate(train_questions["question_text"]):
    inputs = prepare_encoder_inputs(text)

    train_encoder_inputs.extend(inputs)
    train_input_owners.extend([question_position] * len(inputs))

assert len(train_encoder_inputs) == len(train_input_owners)

print("Training questions:", len(train_questions))
print("Encoder inputs:", len(train_encoder_inputs))
print("Ownership entries:", len(train_input_owners))
print("First 10 owners:", train_input_owners[:10])

Training questions: 21900
Encoder inputs: 22021
Ownership entries: 22021
First 10 owners: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [53]:
from time import perf_counter

trial_inputs = train_encoder_inputs[:32]

start_time = perf_counter()

trial_embeddings = encoder.encode(
    trial_inputs,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=False,
)

elapsed_seconds = perf_counter() - start_time

print("Embedding shape:", trial_embeddings.shape)
print("Elapsed seconds:", round(elapsed_seconds, 2))
print(
    "First vector matches earlier result:",
    np.allclose(trial_embeddings[0], short_prompt_embedding, atol=1e-5),
)

Embedding shape: (32, 384)
Elapsed seconds: 0.34
First vector matches earlier result: True


In [54]:
start_time = perf_counter()

train_input_embeddings = encoder.encode(
    train_encoder_inputs,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True,
)

train_encoding_seconds = perf_counter() - start_time

assert train_input_embeddings.shape[0] == len(train_input_owners)

print("Input embedding shape:", train_input_embeddings.shape)
print("Data type:", train_input_embeddings.dtype)
print("All values finite:", np.isfinite(train_input_embeddings).all())
print("Elapsed minutes:", round(train_encoding_seconds / 60, 2))

Batches: 100%|██████████| 2753/2753 [11:25<00:00,  4.02it/s]

Input embedding shape: (22021, 384)
Data type: float32
All values finite: True
Elapsed minutes: 11.42


In [55]:
owners = np.asarray(train_input_owners, dtype=np.int64)

train_chunk_counts = np.bincount(
    owners,
    minlength=len(train_questions),
)

train_embedding_sums = np.zeros(
    (len(train_questions), train_input_embeddings.shape[1]),
    dtype=np.float32,
)

np.add.at(train_embedding_sums, owners, train_input_embeddings)

assert (train_chunk_counts > 0).all()

train_mean_embeddings = (
    train_embedding_sums
    / train_chunk_counts[:, None].astype(np.float32)
)

print("Question embedding shape:", train_mean_embeddings.shape)
print("Data type:", train_mean_embeddings.dtype)
print("Questions by number of inputs:")
print(pd.Series(train_chunk_counts).value_counts().sort_index())

Question embedding shape: (21900, 384)
Data type: float32
Questions by number of inputs:
1    21780
2      119
3        1
Name: count, dtype: int64


In [56]:
train_vector_lengths = np.linalg.norm(
    train_mean_embeddings,
    axis=1,
    keepdims=True,
)

assert np.isfinite(train_vector_lengths).all()
assert (train_vector_lengths > 0).all()

X_train_e5 = train_mean_embeddings / train_vector_lengths

final_lengths = np.linalg.norm(X_train_e5, axis=1)
longest_position = train_questions.index.get_loc(longest_index)

print("Training embedding shape:", X_train_e5.shape)
print("Vector length range:", final_lengths.min(), final_lengths.max())
print(
    "First prompt matches:",
    np.allclose(X_train_e5[0], short_prompt_embedding, atol=1e-5),
)
print(
    "Longest prompt matches:",
    np.allclose(
        X_train_e5[longest_position], long_prompt_embedding, atol=1e-5
    ),
)

Training embedding shape: (21900, 384)
Vector length range: 0.9999999 1.0000001
First prompt matches: True
Longest prompt matches: True


In [57]:
embedding_dir = saved_data_dir.parent / "multilingual_e5_small_chunk480"
embedding_dir.mkdir(parents=True, exist_ok=True)

np.save(embedding_dir / "X_train_e5.npy", X_train_e5)

print("Saved:", (embedding_dir / "X_train_e5.npy").resolve())

Saved: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/multilingual_e5_small_chunk480/X_train_e5.npy


In [58]:
loaded_X_train_e5 = np.load(
    embedding_dir / "X_train_e5.npy"
)

loaded_train_questions = pd.read_pickle(
    saved_data_dir / "train_questions.pkl"
)

assert loaded_X_train_e5.shape == (len(loaded_train_questions), 384)

print("Loaded shape:", loaded_X_train_e5.shape)
print("Loaded data type:", loaded_X_train_e5.dtype)
print(
    "Embedding values unchanged:",
    np.array_equal(loaded_X_train_e5, X_train_e5),
)
print(
    "Question table matches:",
    loaded_train_questions.equals(train_questions),
)

Loaded shape: (21900, 384)
Loaded data type: float32
Embedding values unchanged: True
Question table matches: True


In [59]:
calibration_questions = pd.read_pickle(
    saved_data_dir / "calibration_questions.pkl"
)

calibration_encoder_inputs = []
calibration_input_owners = []

for question_position, text in enumerate(
    calibration_questions["question_text"]
):
    inputs = prepare_encoder_inputs(text)

    calibration_encoder_inputs.extend(inputs)
    calibration_input_owners.extend([question_position] * len(inputs))

assert len(calibration_encoder_inputs) == len(calibration_input_owners)

calibration_chunk_counts = np.bincount(
    calibration_input_owners,
    minlength=len(calibration_questions),
)

print("Calibration questions:", len(calibration_questions))
print("Encoder inputs:", len(calibration_encoder_inputs))
print("Questions by number of inputs:")
print(pd.Series(calibration_chunk_counts).value_counts().sort_index())

Calibration questions: 3648
Encoder inputs: 3681
Questions by number of inputs:
1    3617
2      29
3       2
Name: count, dtype: int64


In [60]:
start_time = perf_counter()

calibration_input_embeddings = encoder.encode(
    calibration_encoder_inputs,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True,
)

calibration_encoding_seconds = perf_counter() - start_time

assert calibration_input_embeddings.shape[0] == len(
    calibration_input_owners
)

print("Input embedding shape:", calibration_input_embeddings.shape)
print("Data type:", calibration_input_embeddings.dtype)
print("All values finite:", np.isfinite(calibration_input_embeddings).all())
print("Elapsed minutes:", round(calibration_encoding_seconds / 60, 2))

Batches: 100%|██████████| 461/461 [01:50<00:00,  4.17it/s]

Input embedding shape: (3681, 384)
Data type: float32
All values finite: True
Elapsed minutes: 1.84


In [61]:
def pool_question_embeddings(input_vectors, input_owners, n_questions):
    owners = np.asarray(input_owners, dtype=np.int64)
    counts = np.bincount(owners, minlength=n_questions)

    assert (counts > 0).all()

    sums = np.zeros(
        (n_questions, input_vectors.shape[1]),
        dtype=np.float32,
    )

    np.add.at(sums, owners, input_vectors)

    means = sums / counts[:, None].astype(np.float32)
    lengths = np.linalg.norm(means, axis=1, keepdims=True)

    assert np.isfinite(lengths).all()
    assert (lengths > 0).all()

    return means / lengths

In [62]:
X_calibration_e5 = pool_question_embeddings(
    calibration_input_embeddings,
    calibration_input_owners,
    len(calibration_questions),
)

calibration_final_lengths = np.linalg.norm(X_calibration_e5, axis=1)

print("Calibration shape:", X_calibration_e5.shape)
print("Data type:", X_calibration_e5.dtype)
print(
    "Vector length range:",
    calibration_final_lengths.min(),
    calibration_final_lengths.max(),
)

Calibration shape: (3648, 384)
Data type: float32
Vector length range: 0.9999999 1.0000001


In [63]:
np.save(
    embedding_dir / "X_calibration_e5.npy",
    X_calibration_e5,
)

loaded_X_calibration_e5 = np.load(
    embedding_dir / "X_calibration_e5.npy"
)

print("Saved:", (embedding_dir / "X_calibration_e5.npy").resolve())
print(
    "Embedding values unchanged:",
    np.array_equal(loaded_X_calibration_e5, X_calibration_e5),
)

Saved: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/multilingual_e5_small_chunk480/X_calibration_e5.npy
Embedding values unchanged: True


In [64]:
def prepare_batch_inputs(question_texts):
    all_inputs = []
    all_owners = []

    for question_position, text in enumerate(question_texts):
        inputs = prepare_encoder_inputs(text)

        all_inputs.extend(inputs)
        all_owners.extend([question_position] * len(inputs))

    return all_inputs, all_owners

validation_questions = pd.read_pickle(
    saved_data_dir / "validation_questions.pkl"
)

validation_encoder_inputs, validation_input_owners = prepare_batch_inputs(
    validation_questions["question_text"]
)

validation_chunk_counts = np.bincount(
    validation_input_owners,
    minlength=len(validation_questions),
)

print("Validation questions:", len(validation_questions))
print("Encoder inputs:", len(validation_encoder_inputs))
print("Questions by number of inputs:")
print(pd.Series(validation_chunk_counts).value_counts().sort_index())

Validation questions: 3649
Encoder inputs: 3671
Questions by number of inputs:
1    3627
2      22
Name: count, dtype: int64


In [65]:
start_time = perf_counter()

validation_input_embeddings = encoder.encode(
    validation_encoder_inputs,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True,
)

validation_encoding_seconds = perf_counter() - start_time

assert validation_input_embeddings.shape[0] == len(
    validation_input_owners
)

print("Input embedding shape:", validation_input_embeddings.shape)
print("All values finite:", np.isfinite(validation_input_embeddings).all())
print("Elapsed minutes:", round(validation_encoding_seconds / 60, 2))

Batches: 100%|██████████| 459/459 [01:46<00:00,  4.30it/s]

Input embedding shape: (3671, 384)
All values finite: True
Elapsed minutes: 1.78


In [66]:
X_validation_e5 = pool_question_embeddings(
    validation_input_embeddings,
    validation_input_owners,
    len(validation_questions),
)

validation_lengths = np.linalg.norm(X_validation_e5, axis=1)

print("Validation shape:", X_validation_e5.shape)
print("Vector length range:", validation_lengths.min(), validation_lengths.max())

np.save(embedding_dir / "X_validation_e5.npy", X_validation_e5)

loaded_X_validation_e5 = np.load(
    embedding_dir / "X_validation_e5.npy"
)

print(
    "Saved values unchanged:",
    np.array_equal(loaded_X_validation_e5, X_validation_e5),
)

Validation shape: (3649, 384)
Vector length range: 0.9999999 1.0000001
Saved values unchanged: True


In [67]:
test_questions = pd.read_pickle(
    saved_data_dir / "test_questions.pkl"
)

test_encoder_inputs, test_input_owners = prepare_batch_inputs(
    test_questions["question_text"]
)

test_chunk_counts = np.bincount(
    test_input_owners,
    minlength=len(test_questions),
)

print("Test questions:", len(test_questions))
print("Encoder inputs:", len(test_encoder_inputs))
print("Questions by number of inputs:")
print(pd.Series(test_chunk_counts).value_counts().sort_index())

Test questions: 7300
Encoder inputs: 7356
Questions by number of inputs:
1    7246
2      52
3       2
Name: count, dtype: int64


In [68]:
start_time = perf_counter()

test_input_embeddings = encoder.encode(
    test_encoder_inputs,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True,
)

test_encoding_seconds = perf_counter() - start_time

assert test_input_embeddings.shape[0] == len(test_input_owners)

print("Input embedding shape:", test_input_embeddings.shape)
print("All values finite:", np.isfinite(test_input_embeddings).all())
print("Elapsed minutes:", round(test_encoding_seconds / 60, 2))

Batches: 100%|██████████| 920/920 [03:35<00:00,  4.26it/s]

Input embedding shape: (7356, 384)
All values finite: True
Elapsed minutes: 3.6


In [69]:
X_test_e5 = pool_question_embeddings(
    test_input_embeddings,
    test_input_owners,
    len(test_questions),
)

test_lengths = np.linalg.norm(X_test_e5, axis=1)

print("Test shape:", X_test_e5.shape)
print("Vector length range:", test_lengths.min(), test_lengths.max())

np.save(embedding_dir / "X_test_e5.npy", X_test_e5)

loaded_X_test_e5 = np.load(
    embedding_dir / "X_test_e5.npy"
)

print(
    "Saved values unchanged:",
    np.array_equal(loaded_X_test_e5, X_test_e5),
)

Test shape: (7300, 384)
Vector length range: 0.9999999 1.0000001
Saved values unchanged: True


In [70]:
import json
from importlib.metadata import version

embedding_metadata = {
    "model_name": embedding_model_name,
    "embedding_dimension": 384,
    "max_input_tokens": encoder.max_seq_length,
    "input_prefix": "query: ",
    "long_prompt_chunk_tokens": 480,
    "chunk_overlap_tokens": 0,
    "chunking_method": "split content token IDs, then decode each chunk",
    "pooling": "equal-weight mean of chunk vectors",
    "normalize_each_input": True,
    "normalize_after_pooling": True,
    "dtype": str(X_train_e5.dtype),
    "question_tables_directory": str(saved_data_dir.resolve()),
    "row_order": "same positional order as the corresponding question table",
    "package_versions": {
        name: version(name)
        for name in [
            "sentence-transformers",
            "transformers",
            "tokenizers",
            "torch",
            "numpy",
            "pandas",
            "pyarrow",
        ]
    },
}

metadata_path = embedding_dir / "embedding_metadata.json"

metadata_path.write_text(
    json.dumps(embedding_metadata, indent=2),
    encoding="utf-8",
)

print("Saved metadata:", metadata_path.resolve())

Saved metadata: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/multilingual_e5_small_chunk480/embedding_metadata.json


In [71]:
import joblib

tfidf_preprocessing = joblib.load(
    saved_data_dir / "preprocessing.joblib"
)

length_scaler = tfidf_preprocessing["length_scaler"]

print("Scaler feature:", length_scaler.feature_names_in_.tolist())
print("Training mean length:", length_scaler.mean_[0])
print("Training length scale:", length_scaler.scale_[0])

Scaler feature: ['char_length']
Training mean length: 761.3295890410959
Training length scale: 429.62121380676626


In [72]:
import numpy as np
train_length_features = (
    train_questions["question_text"]
    .str.len()
    .to_frame("char_length")
)

train_length_scaled = length_scaler.transform(
    train_length_features
).astype(np.float32)

X_train_e5_features = np.hstack(
    [X_train_e5, train_length_scaled]
)

print("Scaled length shape:", train_length_scaled.shape)
print("Combined feature shape:", X_train_e5_features.shape)
print("Data type:", X_train_e5_features.dtype)

Scaled length shape: (21900, 1)
Combined feature shape: (21900, 385)
Data type: float32


In [73]:
calibration_length_features = (
    calibration_questions["question_text"]
    .str.len()
    .to_frame("char_length")
)

calibration_length_scaled = length_scaler.transform(
    calibration_length_features
).astype(np.float32)

X_calibration_e5_features = np.hstack(
    [X_calibration_e5, calibration_length_scaled]
)

print("Scaled length shape:", calibration_length_scaled.shape)
print("Combined feature shape:", X_calibration_e5_features.shape)
print("Data type:", X_calibration_e5_features.dtype)

Scaled length shape: (3648, 1)
Combined feature shape: (3648, 385)
Data type: float32


In [74]:
validation_length_features = (
    validation_questions["question_text"]
    .str.len()
    .to_frame("char_length")
)

validation_length_scaled = length_scaler.transform(
    validation_length_features
).astype(np.float32)

X_validation_e5_features = np.hstack(
    [X_validation_e5, validation_length_scaled]
)

print("Scaled length shape:", validation_length_scaled.shape)
print("Combined feature shape:", X_validation_e5_features.shape)
print("Data type:", X_validation_e5_features.dtype)

Scaled length shape: (3649, 1)
Combined feature shape: (3649, 385)
Data type: float32


In [75]:
test_length_features = (
    test_questions["question_text"]
    .str.len()
    .to_frame("char_length")
)

test_length_scaled = length_scaler.transform(
    test_length_features
).astype(np.float32)

X_test_e5_features = np.hstack(
    [X_test_e5, test_length_scaled]
)

print("Scaled length shape:", test_length_scaled.shape)
print("Combined feature shape:", X_test_e5_features.shape)
print("Data type:", X_test_e5_features.dtype)

Scaled length shape: (7300, 1)
Combined feature shape: (7300, 385)
Data type: float32


In [76]:
feature_sets = {
    "train": (X_train_e5_features, X_train_e5),
    "calibration": (X_calibration_e5_features, X_calibration_e5),
    "validation": (X_validation_e5_features, X_validation_e5),
    "test": (X_test_e5_features, X_test_e5),
}

for split_name, (features, embeddings) in feature_sets.items():
    assert features.shape == (embeddings.shape[0], 385)
    assert features.dtype == np.float32
    assert np.isfinite(features).all()
    assert np.array_equal(features[:, :384], embeddings)

    print(split_name, "checks passed:", features.shape)

train checks passed: (21900, 385)
calibration checks passed: (3648, 385)
validation checks passed: (3649, 385)
test checks passed: (7300, 385)


In [77]:
train_features_path = embedding_dir / "X_train_e5_features.npy"

np.save(train_features_path, X_train_e5_features)

loaded_train_features = np.load(train_features_path)

print("Saved:", train_features_path.resolve())
print("Loaded shape:", loaded_train_features.shape)
print(
    "Values unchanged:",
    np.array_equal(loaded_train_features, X_train_e5_features),
)

Saved: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/multilingual_e5_small_chunk480/X_train_e5_features.npy
Loaded shape: (21900, 385)
Values unchanged: True


In [78]:
calibration_features_path = embedding_dir / "X_calibration_e5_features.npy"

np.save(calibration_features_path, X_calibration_e5_features)

loaded_calibration_features = np.load(calibration_features_path)

print("Saved:", calibration_features_path.resolve())
print("Loaded shape:", loaded_calibration_features.shape)
print(
    "Values unchanged:",
    np.array_equal(
        loaded_calibration_features,
        X_calibration_e5_features,
    ),
)

Saved: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/multilingual_e5_small_chunk480/X_calibration_e5_features.npy
Loaded shape: (3648, 385)
Values unchanged: True


In [79]:
validation_features_path = embedding_dir / "X_validation_e5_features.npy"

np.save(validation_features_path, X_validation_e5_features)

loaded_validation_features = np.load(validation_features_path)

print("Saved:", validation_features_path.resolve())
print("Loaded shape:", loaded_validation_features.shape)
print(
    "Values unchanged:",
    np.array_equal(
        loaded_validation_features,
        X_validation_e5_features,
    ),
)

Saved: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/multilingual_e5_small_chunk480/X_validation_e5_features.npy
Loaded shape: (3649, 385)
Values unchanged: True


In [80]:
test_features_path = embedding_dir / "X_test_e5_features.npy"

np.save(test_features_path, X_test_e5_features)

loaded_test_features = np.load(test_features_path)

print("Saved:", test_features_path.resolve())
print("Loaded shape:", loaded_test_features.shape)
print(
    "Values unchanged:",
    np.array_equal(loaded_test_features, X_test_e5_features),
)

Saved: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/multilingual_e5_small_chunk480/X_test_e5_features.npy
Loaded shape: (7300, 385)
Values unchanged: True


In [81]:
scaler_path = embedding_dir / "length_scaler.joblib"

joblib.dump(length_scaler, scaler_path)

loaded_length_scaler = joblib.load(scaler_path)

recreated_train_lengths = loaded_length_scaler.transform(
    train_length_features
).astype(np.float32)

print("Saved:", scaler_path.resolve())
print(
    "Scaled lengths unchanged:",
    np.array_equal(recreated_train_lengths, train_length_scaled),
)

Saved: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/multilingual_e5_small_chunk480/length_scaler.joblib
Scaled lengths unchanged: True


In [82]:
embedding_metadata.update({
    "combined_feature_dimension": 385,
    "combined_feature_order": [
        "normalized_e5_embedding: columns 0–383",
        "standardized_char_length: column 384",
    ],
    "length_scaler_file": "length_scaler.joblib",
    "combined_feature_filename_pattern": "X_{split}_e5_features.npy",
})

metadata_path.write_text(
    json.dumps(embedding_metadata, indent=2),
    encoding="utf-8",
)

print("Updated metadata:", metadata_path.resolve())
print("Combined dimension:", embedding_metadata["combined_feature_dimension"])

Updated metadata: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/multilingual_e5_small_chunk480/embedding_metadata.json
Combined dimension: 385


In [83]:
model_names = tfidf_preprocessing["model_names"]

train_scores = pd.read_pickle(saved_data_dir / "train_scores.pkl")
train_costs = pd.read_pickle(saved_data_dir / "train_costs.pkl")

assert (
    loaded_train_features.shape[0]
    == len(train_questions)
    == len(train_scores)
    == len(train_costs)
)

assert train_questions.index.equals(train_scores.index)
assert train_questions.index.equals(train_costs.index)

assert train_scores.columns.tolist() == model_names
assert train_costs.columns.tolist() == model_names

print("Training alignment checks passed.")
print("Features:", loaded_train_features.shape)
print("Scores:", train_scores.shape)
print("Costs:", train_costs.shape)

Training alignment checks passed.
Features: (21900, 385)
Scores: (21900, 11)
Costs: (21900, 11)


In [84]:
calibration_scores = pd.read_pickle(
    saved_data_dir / "calibration_scores.pkl"
)
calibration_costs = pd.read_pickle(
    saved_data_dir / "calibration_costs.pkl"
)

assert (
    loaded_calibration_features.shape[0]
    == len(calibration_questions)
    == len(calibration_scores)
    == len(calibration_costs)
)

assert calibration_questions.index.equals(calibration_scores.index)
assert calibration_questions.index.equals(calibration_costs.index)

assert calibration_scores.columns.tolist() == model_names
assert calibration_costs.columns.tolist() == model_names

print("Calibration alignment checks passed.")
print("Features:", loaded_calibration_features.shape)
print("Scores:", calibration_scores.shape)
print("Costs:", calibration_costs.shape)

Calibration alignment checks passed.
Features: (3648, 385)
Scores: (3648, 11)
Costs: (3648, 11)


In [85]:
validation_scores = pd.read_pickle(
    saved_data_dir / "validation_scores.pkl"
)
validation_costs = pd.read_pickle(
    saved_data_dir / "validation_costs.pkl"
)

assert (
    loaded_validation_features.shape[0]
    == len(validation_questions)
    == len(validation_scores)
    == len(validation_costs)
)

assert validation_questions.index.equals(validation_scores.index)
assert validation_questions.index.equals(validation_costs.index)

assert validation_scores.columns.tolist() == model_names
assert validation_costs.columns.tolist() == model_names

print("Validation alignment checks passed.")
print("Features:", loaded_validation_features.shape)
print("Scores:", validation_scores.shape)
print("Costs:", validation_costs.shape)

Validation alignment checks passed.
Features: (3649, 385)
Scores: (3649, 11)
Costs: (3649, 11)


In [86]:
test_scores = pd.read_pickle(
    saved_data_dir / "test_scores.pkl"
)
test_costs = pd.read_pickle(
    saved_data_dir / "test_costs.pkl"
)

assert (
    loaded_test_features.shape[0]
    == len(test_questions)
    == len(test_scores)
    == len(test_costs)
)

assert test_questions.index.equals(test_scores.index)
assert test_questions.index.equals(test_costs.index)

assert test_scores.columns.tolist() == model_names
assert test_costs.columns.tolist() == model_names

print("Test alignment checks passed.")
print("Features:", loaded_test_features.shape)
print("Scores:", test_scores.shape)
print("Costs:", test_costs.shape)

Test alignment checks passed.
Features: (7300, 385)
Scores: (7300, 11)
Costs: (7300, 11)


In [87]:
multiple_choice_mask = train_questions["eval_name"].str.startswith(
    "mmlu",
    na=False,
)

choice_example = train_questions.loc[multiple_choice_mask].iloc[0]

print("Sample ID:", choice_example["sample_id"])
print("Full prompt:")
print(choice_example["question_text"])

Sample ID: mmlu-abstract-algebra.val.10
Full prompt:
Please answer with the letter of the correct answer.

Statement 1 | If H is a subgroup of G and a belongs to G then |aH| = |Ha|. Statement 2 | If H is a subgroup of G and a and b belong to G, then aH and Hb are identical or disjoint.
A) False, False
B) True, False
C) False, True
D) True, True
Print only a single choice  from "A" or "B" or "C" or "D" without explanation. Answer:


In [88]:
import re

choice_pattern = r"(?<!\S)([A-Z])\)\s+"

choice_labels = re.findall(
    choice_pattern,
    choice_example["question_text"],
)

print("Detected labels:", choice_labels)
print("Answer-choice count:", len(choice_labels))

Detected labels: ['A', 'B', 'C', 'D']
Answer-choice count: 4


In [89]:
train_choice_labels = train_questions["question_text"].apply(
    lambda text: re.findall(choice_pattern, text)
)

train_choice_match_counts = train_choice_labels.str.len()

print("Number of prompts by detected label count:")
print(train_choice_match_counts.value_counts().sort_index())

Number of prompts by detected label count:
question_text
0      5810
1         3
2       786
3         3
4     15257
5        21
6         5
7         8
8         3
10        1
11        1
12        1
13        1
Name: count, dtype: int64


In [90]:
for match_count in [1, 3, 13]:
    matching_rows = train_choice_match_counts.eq(match_count)
    example_index = train_choice_match_counts.loc[matching_rows].index[0]

    example = train_questions.loc[example_index]

    print("\nDetected count:", match_count)
    print("Sample ID:", example["sample_id"])
    print("Labels:", train_choice_labels.loc[example_index])
    print("Prompt:")
    print(example["question_text"])


Detected count: 1
Sample ID: abstract2title.test.11
Labels: ['I']
Prompt:
please write an title based on this abstract of article: 

Organic-inorganic hybrid perovskites have recently attracted substantial attention as a top candidate for use as light-absorbing materials in high-efficiency, low-cost and solution-processable photovoltaic devices owing to their excellent optoelectronic properties. Here, we fabricated inverted planar perovskite solar cells by incorporating small amounts of ammonium halide NH4X (X=F, Cl, Br, I) additives into a CH3NH3PbI3 (MAPbI3) perovskite solution. A compact and uniform perovskite absorber layer with large perovskite crystalline grains is realized by simply incorporating small amounts of additives and by using an anti-solvent engineering technique to control the nucleation and crystal growth of perovskite. The enlarged perovskite grain size with a reduced density of the grain boundaries and improved crystallinity results in fewer charge carrier recombi

In [91]:
formula_mask = train_questions["sample_id"].eq(
    "mmlu-machine-learning.val.97"
)
formula_text = train_questions.loc[formula_mask, "question_text"].iloc[0]

for match in re.finditer(choice_pattern, formula_text):
    start = max(0, match.start() - 25)
    end = min(len(formula_text), match.end() + 35)

    print(match.group(1), "→", repr(formula_text[start:end]))

C → 'wing is equal to P(A, B, C) given Boolean random variables A, B'
A → 'ons between any of them?\nA) P(A | B) * P(B | C) * P(C | A)\nB) P'
B → 'en any of them?\nA) P(A | B) * P(B | C) * P(C | A)\nB) P(C | A, B'
C → 'hem?\nA) P(A | B) * P(B | C) * P(C | A)\nB) P(C | A, B) * P(A) * '
A → ' | B) * P(B | C) * P(C | A)\nB) P(C | A, B) * P(A) * P(B)\nC) P(A'
B → 'B) * P(B | C) * P(C | A)\nB) P(C | A, B) * P(A) * P(B)\nC) P(A | '
B → ') * P(C | A)\nB) P(C | A, B) * P(A) * P(B)\nC) P(A | B, C) * P(B '
C → '(C | A, B) * P(A) * P(B)\nC) P(A | B, C) * P(B | A, C) * P(C | A'
C → ' P(A) * P(B)\nC) P(A | B, C) * P(B | A, C) * P(C | A, B)\nD) P(A,'
C → ') P(A | B, C) * P(B | A, C) * P(C | A, B)\nD) P(A, B | C) * P(C)'
B → '* P(B | A, C) * P(C | A, B)\nD) P(A, B | C) * P(C)\nPrint only a '
D → '(B | A, C) * P(C | A, B)\nD) P(A, B | C) * P(C)\nPrint only a sin'
C → ' P(C | A, B)\nD) P(A, B | C) * P(C)\nPrint only a single choice  '


In [92]:
formula_mask = train_questions["sample_id"].eq(
    "mmlu-machine-learning.val.97"
)
formula_text = train_questions.loc[formula_mask, "question_text"].iloc[0]

for match in re.finditer(choice_pattern, formula_text):
    start = max(0, match.start() - 25)
    end = min(len(formula_text), match.end() + 35)

    print(match.group(1), "→", repr(formula_text[start:end]))

C → 'wing is equal to P(A, B, C) given Boolean random variables A, B'
A → 'ons between any of them?\nA) P(A | B) * P(B | C) * P(C | A)\nB) P'
B → 'en any of them?\nA) P(A | B) * P(B | C) * P(C | A)\nB) P(C | A, B'
C → 'hem?\nA) P(A | B) * P(B | C) * P(C | A)\nB) P(C | A, B) * P(A) * '
A → ' | B) * P(B | C) * P(C | A)\nB) P(C | A, B) * P(A) * P(B)\nC) P(A'
B → 'B) * P(B | C) * P(C | A)\nB) P(C | A, B) * P(A) * P(B)\nC) P(A | '
B → ') * P(C | A)\nB) P(C | A, B) * P(A) * P(B)\nC) P(A | B, C) * P(B '
C → '(C | A, B) * P(A) * P(B)\nC) P(A | B, C) * P(B | A, C) * P(C | A'
C → ' P(A) * P(B)\nC) P(A | B, C) * P(B | A, C) * P(C | A, B)\nD) P(A,'
C → ') P(A | B, C) * P(B | A, C) * P(C | A, B)\nD) P(A, B | C) * P(C)'
B → '* P(B | A, C) * P(C | A, B)\nD) P(A, B | C) * P(C)\nPrint only a '
D → '(B | A, C) * P(C | A, B)\nD) P(A, B | C) * P(C)\nPrint only a sin'
C → ' P(C | A, B)\nD) P(A, B | C) * P(C)\nPrint only a single choice  '


In [93]:
line_choice_pattern = r"(?m)^[ \t]*([A-Z])\)[ \t]+"

example_ids = [
    "mmlu-abstract-algebra.val.10",
    "arc-challenge.test.120",
    "mmlu-machine-learning.val.97",
    "abstract2title.test.11",
]

for sample_id in example_ids:
    text = train_questions.loc[
        train_questions["sample_id"].eq(sample_id),
        "question_text",
    ].iloc[0]

    labels = re.findall(line_choice_pattern, text)
    print(sample_id, "→", labels, "Count:", len(labels))

mmlu-abstract-algebra.val.10 → ['A', 'B', 'C', 'D'] Count: 4
arc-challenge.test.120 → ['A', 'B', 'C'] Count: 3
mmlu-machine-learning.val.97 → ['A', 'B', 'C', 'D'] Count: 4
abstract2title.test.11 → [] Count: 0


In [94]:
def count_answer_choices(text):
    labels = re.findall(line_choice_pattern, text)

    if len(labels) < 2:
        return 0

    expected_labels = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ"[:len(labels)])

    if labels != expected_labels:
        return 0

    return len(labels)


train_answer_choice_counts = train_questions["question_text"].apply(
    count_answer_choices
)

print("Training prompts by detected answer-choice count:")
print(train_answer_choice_counts.value_counts().sort_index())


Training prompts by detected answer-choice count:
question_text
0     5813
2      786
3        3
4    15297
5        1
Name: count, dtype: int64


In [95]:
choice_audit = pd.DataFrame({
    "eval_name": train_questions["eval_name"],
    "detected": train_answer_choice_counts.gt(0),
})

choice_coverage = choice_audit.groupby("eval_name").agg(
    prompts=("detected", "size"),
    detected=("detected", "sum"),
)

choice_coverage["undetected"] = (
    choice_coverage["prompts"] - choice_coverage["detected"]
)

display(
    choice_coverage.loc[choice_coverage["undetected"] > 0]
    .sort_values("undetected", ascending=False)
    .head(15)
)


,prompts,detected,undetected
eval_name,,,
grade-school-math,4492,0,4492
mbpp,260,0,260
chinese_zodiac,239,0,239
consensus_summary,213,0,213
bias_detection,153,0,153
abstract2title,145,0,145
Chinese_character_riddles,62,0,62
mtbench,27,0,27
chinese_tang_poetries,19,0,19


In [96]:
five_choice_index = train_answer_choice_counts.loc[
    train_answer_choice_counts.eq(5)
].index[0]

five_choice_example = train_questions.loc[five_choice_index]

print("Sample ID:", five_choice_example["sample_id"])
print("Prompt:")
print(five_choice_example["question_text"])

Sample ID: arc-challenge.test.853
Prompt:
Sally placed electrodes into a beaker containing a solution and connected the electrodes to a batter. Part of Sally's report stated "Bubbles were given off at one of the electrodes." The statement is
A) a prediction
B) a hypothesis
C) an observation
D) a theory
E) a conclusion
Print only a single choice  from "A" or "B" or "C" or "D" without explanation.
Answer:


In [97]:
train_handcrafted_features = train_length_features.copy()

train_handcrafted_features["answer_choice_count"] = (
    train_answer_choice_counts
)

display(train_handcrafted_features.head())
print("Feature table shape:", train_handcrafted_features.shape)

,char_length,answer_choice_count
0,101,0
3,93,0
4,89,0
5,81,0
6,93,0


Feature table shape: (21900, 2)


In [98]:
numeric_pattern = r"\d+(?:,\d{3})*(?:\.\d+)?"

numeric_examples = [
    "Solve 2x + 3 = 7.",
    "The price is 12.50 and the tax is 0.75.",
    "There are 1,200 students.",
    "Explain photosynthesis.",
]

for text in numeric_examples:
    numbers = re.findall(numeric_pattern, text)
    print(text)
    print("Detected numbers:", numbers, "Count:", len(numbers))

Solve 2x + 3 = 7.
Detected numbers: ['2', '3', '7'] Count: 3
The price is 12.50 and the tax is 0.75.
Detected numbers: ['12.50', '0.75'] Count: 2
There are 1,200 students.
Detected numbers: ['1,200'] Count: 1
Explain photosynthesis.
Detected numbers: [] Count: 0


In [99]:
def count_numeric_values(text):
    return len(re.findall(numeric_pattern, text))


train_handcrafted_features["numeric_value_count"] = (
    train_questions["question_text"].apply(count_numeric_values)
)

display(train_handcrafted_features.head())

print("Numeric-count summary:")
print(train_handcrafted_features["numeric_value_count"].describe())

print("Feature table shape:", train_handcrafted_features.shape)

,char_length,answer_choice_count,numeric_value_count
0,101,0,0
3,93,0,0
4,89,0,0
5,81,0,0
6,93,0,0


Numeric-count summary:
count    21900.000000
mean         4.662009
std          7.042557
min          0.000000
25%          0.000000
50%          0.000000
75%          7.000000
max         49.000000
Name: numeric_value_count, dtype: float64
Feature table shape: (21900, 3)


In [100]:
code_example = train_questions.loc[
    train_questions["eval_name"].eq("mbpp")
].iloc[0]

print("Sample ID:", code_example["sample_id"])
print("Prompt:")
print(code_example["question_text"])

Sample ID: mbpp.dev.100
Prompt:
Write a function to find the sum of numbers in a list within a range specified by two indices.


In [101]:
programming_request_pattern = (
    r"\b(?:write|implement|debug|fix|complete|create)\b"
    r"[^\n.!?]{0,100}"
    r"\b(?:function|code|program|script)\b"
)

code_examples = [
    code_example["question_text"],
    "Debug this Python code.",
    "Explain the function of the heart.",
    "Write a poem about the ocean.",
]

for text in code_examples:
    detected = bool(
        re.search(programming_request_pattern, text, flags=re.IGNORECASE)
    )
    print(detected, "→", text)

True → Write a function to find the sum of numbers in a list within a range specified by two indices.
True → Debug this Python code.
False → Explain the function of the heart.
False → Write a poem about the ocean.


In [102]:
code_syntax_pattern = (
    r"(?m)^[ \t]*(?:def[ \t]+\w+[ \t]*\("
    r"|import[ \t]+\w+|assert[ \t]+)"
)

chinese_programming_pattern = (
    r"(?:编写|实现|调试|修复|补全|写)"
    r"[^\n。！？]{0,60}"
    r"(?:函数|代码|程序|脚本)"
)

code_patterns = [
    r"```",
    code_syntax_pattern,
    programming_request_pattern,
    chinese_programming_pattern,
]


def detect_code(text):
    detected = any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in code_patterns
    )
    return int(detected)


new_code_examples = [
    "def add(a, b):\n    return a + b",
    "```python\nprint(1)\n```",
    "请编写一个计算列表总和的函数。",
    "请解释光合作用。",
]

for text in new_code_examples:
    print(detect_code(text), "→", repr(text))

1 → 'def add(a, b):\n    return a + b'
1 → '```python\nprint(1)\n```'
1 → '请编写一个计算列表总和的函数。'
0 → '请解释光合作用。'


In [103]:
train_handcrafted_features["code_indicator"] = (
    train_questions["question_text"].apply(detect_code)
)

display(train_handcrafted_features.head())

print("Training prompts by code indicator:")
print(
    train_handcrafted_features["code_indicator"]
    .value_counts()
    .sort_index()
)

print("Feature table shape:", train_handcrafted_features.shape)

,char_length,answer_choice_count,numeric_value_count,code_indicator
0,101,0,0,0
3,93,0,0,0
4,89,0,0,0
5,81,0,0,0
6,93,0,0,0


Training prompts by code indicator:
code_indicator
0    21625
1      275
Name: count, dtype: int64
Feature table shape: (21900, 4)


In [104]:
mbpp_mask = train_questions["eval_name"].eq("mbpp")

print("MBPP prompts:", mbpp_mask.sum())
print(
    "MBPP prompts flagged:",
    train_handcrafted_features.loc[mbpp_mask, "code_indicator"].sum(),
)

other_code_mask = (
    train_handcrafted_features["code_indicator"].eq(1)
    & ~mbpp_mask
)

other_code_example = train_questions.loc[other_code_mask].iloc[0]

print("\nExample outside MBPP:")
print("Sample ID:", other_code_example["sample_id"])
print(other_code_example["question_text"])

MBPP prompts: 260
MBPP prompts flagged: 260

Example outside MBPP:
Sample ID: hellaswag.val.4669
[header] How to password protect a web page [title] Create or copy the basic code. [step] This method is best if you are hosting your own page or are simply trying to learn more about password protection coding. There are many different variations on code for basic password protection (we've reprinted one), and we will explain how to set your own password and customize the code in a later step.
A) [title] Locate keywords-nowadays, any keyword-based code is password protected. [step] Keywords are found in the search engine under " options ".
B) [substeps] Copy the basic code for your web page or your own page that you used to create it. Make sure that the code is easy to remember.
C) [title] Add it to your page's code. [step] In order to implement this code you will need editing access to your web page's underlying code.
D) [title] Find the code area of the page. [step] Find the space that s

In [105]:
code_content_patterns = [r"```", code_syntax_pattern]

code_request_patterns = [
    programming_request_pattern,
    chinese_programming_pattern,
]


def detect_code(text):
    has_code_content = any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in code_content_patterns
    )

    if has_code_content:
        return 1

    if count_answer_choices(text) > 0:
        return 0

    has_programming_request = any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in code_request_patterns
    )

    return int(has_programming_request)


print("HellaSwag passage:", detect_code(other_code_example["question_text"]))
print("MBPP request:", detect_code(code_example["question_text"]))

code_choice_example = (
    "What does this code print?\n"
    "```python\nprint(1 + 1)\n```\n"
    "A) 1\nB) 2"
)
print("Multiple-choice with code:", detect_code(code_choice_example))

HellaSwag passage: 0
MBPP request: 1
Multiple-choice with code: 1


In [106]:
train_handcrafted_features["code_indicator"] = (
    train_questions["question_text"].apply(detect_code)
)

print("Updated code-indicator counts:")
print(
    train_handcrafted_features["code_indicator"]
    .value_counts()
    .sort_index()
)

print(
    "MBPP prompts flagged:",
    train_handcrafted_features.loc[mbpp_mask, "code_indicator"].sum(),
)

print("Feature table shape:", train_handcrafted_features.shape)

Updated code-indicator counts:
code_indicator
0    21634
1      266
Name: count, dtype: int64
MBPP prompts flagged: 260
Feature table shape: (21900, 4)


In [107]:
def build_handcrafted_features(question_table):
    texts = question_table["question_text"]

    features = pd.DataFrame(index=question_table.index)

    features["char_length"] = texts.str.len()
    features["answer_choice_count"] = texts.apply(count_answer_choices)
    features["numeric_value_count"] = texts.apply(count_numeric_values)
    features["code_indicator"] = texts.apply(detect_code)

    return features


calibration_handcrafted_features = build_handcrafted_features(
    calibration_questions
)

display(calibration_handcrafted_features.head())
print("Calibration feature shape:", calibration_handcrafted_features.shape)

,char_length,answer_choice_count,numeric_value_count,code_indicator
11,81,0,0,0
15,84,0,0,0
20,81,0,0,0
36,92,0,0,0
39,81,0,0,0


Calibration feature shape: (3648, 4)


In [108]:
validation_handcrafted_features = build_handcrafted_features(
    validation_questions
)

test_handcrafted_features = build_handcrafted_features(
    test_questions
)

print("Validation feature shape:", validation_handcrafted_features.shape)
print("Test feature shape:", test_handcrafted_features.shape)

Validation feature shape: (3649, 4)
Test feature shape: (7300, 4)


In [109]:
from sklearn.preprocessing import StandardScaler

count_feature_columns = [
    "answer_choice_count",
    "numeric_value_count",
]

count_scaler = StandardScaler()

train_counts_scaled = count_scaler.fit_transform(
    train_handcrafted_features[count_feature_columns]
).astype(np.float32)

print("Scaled training counts shape:", train_counts_scaled.shape)
print("Data type:", train_counts_scaled.dtype)

display(pd.DataFrame(
    {
        "training_mean": count_scaler.mean_,
        "training_scale": count_scaler.scale_,
    },
    index=count_feature_columns,
))

Scaled training counts shape: (21900, 2)
Data type: float32


,training_mean,training_scale
answer_choice_count,2.866393,1.762277
numeric_value_count,4.662009,7.042397


In [110]:
calibration_counts_scaled = count_scaler.transform(
    calibration_handcrafted_features[count_feature_columns]
).astype(np.float32)

validation_counts_scaled = count_scaler.transform(
    validation_handcrafted_features[count_feature_columns]
).astype(np.float32)

test_counts_scaled = count_scaler.transform(
    test_handcrafted_features[count_feature_columns]
).astype(np.float32)

print("Calibration:", calibration_counts_scaled.shape)
print("Validation:", validation_counts_scaled.shape)
print("Test:", test_counts_scaled.shape)

Calibration: (3648, 2)
Validation: (3649, 2)
Test: (7300, 2)


In [111]:
train_code_indicator = train_handcrafted_features[
    ["code_indicator"]
].to_numpy(dtype=np.float32)

X_train_e5_full_features = np.hstack([
    X_train_e5_features,
    train_counts_scaled,
    train_code_indicator,
])

print("Code-indicator shape:", train_code_indicator.shape)
print("Full training shape:", X_train_e5_full_features.shape)
print("Data type:", X_train_e5_full_features.dtype)

Code-indicator shape: (21900, 1)
Full training shape: (21900, 388)
Data type: float32


In [112]:
calibration_code_indicator = calibration_handcrafted_features[
    ["code_indicator"]
].to_numpy(dtype=np.float32)

X_calibration_e5_full_features = np.hstack([
    X_calibration_e5_features,
    calibration_counts_scaled,
    calibration_code_indicator,
])

validation_code_indicator = validation_handcrafted_features[
    ["code_indicator"]
].to_numpy(dtype=np.float32)

X_validation_e5_full_features = np.hstack([
    X_validation_e5_features,
    validation_counts_scaled,
    validation_code_indicator,
])

test_code_indicator = test_handcrafted_features[
    ["code_indicator"]
].to_numpy(dtype=np.float32)

X_test_e5_full_features = np.hstack([
    X_test_e5_features,
    test_counts_scaled,
    test_code_indicator,
])

print("Calibration:", X_calibration_e5_full_features.shape)
print("Validation:", X_validation_e5_full_features.shape)
print("Test:", X_test_e5_full_features.shape)

Calibration: (3648, 388)
Validation: (3649, 388)
Test: (7300, 388)


In [113]:
full_feature_matrices = {
    "train": X_train_e5_full_features,
    "calibration": X_calibration_e5_full_features,
    "validation": X_validation_e5_full_features,
    "test": X_test_e5_full_features,
}

for split_name, matrix in full_feature_matrices.items():
    assert matrix.shape[1] == 388
    assert matrix.dtype == np.float32
    assert np.isfinite(matrix).all()

    file_path = embedding_dir / f"X_{split_name}_e5_full_features.npy"
    np.save(file_path, matrix)

    restored_matrix = np.load(file_path)

    print(
        split_name,
        restored_matrix.shape,
        "Values unchanged:",
        np.array_equal(restored_matrix, matrix),
    )

train (21900, 388) Values unchanged: True
calibration (3648, 388) Values unchanged: True
validation (3649, 388) Values unchanged: True
test (7300, 388) Values unchanged: True


In [114]:
count_scaler_path = embedding_dir / "count_scaler.joblib"

joblib.dump(count_scaler, count_scaler_path)

loaded_count_scaler = joblib.load(count_scaler_path)

recreated_train_counts = loaded_count_scaler.transform(
    train_handcrafted_features[count_feature_columns]
).astype(np.float32)

print("Saved:", count_scaler_path.resolve())
print(
    "Scaled counts unchanged:",
    np.array_equal(recreated_train_counts, train_counts_scaled),
)

Saved: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/multilingual_e5_small_chunk480/count_scaler.joblib
Scaled counts unchanged: True


In [115]:
embedding_metadata.update({
    "full_feature_dimension": 388,
    "full_feature_filename_pattern": "X_{split}_e5_full_features.npy",
    "full_feature_order": [
        "normalized_e5_embedding: columns 0–383",
        "standardized_char_length: column 384",
        "standardized_answer_choice_count: column 385",
        "standardized_numeric_value_count: column 386",
        "binary_code_indicator: column 387",
    ],
    "count_scaler_file": "count_scaler.joblib",
    "count_scaler_columns": count_feature_columns,
    "answer_choice_pattern": line_choice_pattern,
    "answer_choice_validation": "at least two labels, consecutive from A",
    "numeric_value_pattern": numeric_pattern,
    "code_content_patterns": code_content_patterns,
    "code_request_patterns": code_request_patterns,
    "code_regex_flags": "IGNORECASE",
    "code_detection_priority": [
        "code-content match -> 1",
        "otherwise accepted answer choices -> 0",
        "otherwise programming-request match -> 1",
        "otherwise -> 0",
    ],
})

metadata_path.write_text(
    json.dumps(embedding_metadata, indent=2),
    encoding="utf-8",
)

print("Updated metadata:", metadata_path.resolve())
print("Full feature dimension:", embedding_metadata["full_feature_dimension"])

Updated metadata: /Users/shaoguanhua/Desktop/Erdos-Router/1_data_preparation/data/processed/multilingual_e5_small_chunk480/embedding_metadata.json
Full feature dimension: 388
